# Assignment 2: The Dirty Ledger
**Input file:** `teaching_prices_messy.csv`  
**Method:** The raw data is loaded and preserved. A separate working copy is used for diagnostics and type checks. Potential problems are flagged for inspection; no rows are deleted and no values are imputed automatically.

## Diagnostic approach

The code reports the structure and data types, descriptive statistics, missing values, exact duplicates, repeated date-item-city keys, invalid price/date conversions, category labels, units, source labels, and the number of observations per item.

The original missing price and a non-numeric price are counted separately. A conversion failure is not treated as an originally missing value. The code also displays the affected records so that decisions can be made using evidence.

## Planned treatment

- **Investigate and flag:** missing values, non-numeric prices, exact duplicates, repeated keys, and unusual prices.
- **Retain unless evidence supports a change:** unusual but potentially genuine observations.
- **Do not impute or delete automatically:** no mean-filling or blanket `dropna`.
- **Preserve:** the raw input, units, and illustrative-data source label.
- **Review:** capitalization/spacing differences in item labels before standardizing them.

The notebook is diagnostic: it does not claim that flagged records have been corrected. Any later correction should be documented and compared with the unchanged raw data.


In [2]:
import pandas as pd

source_name = "teaching_prices_messy.csv"
raw = pd.read_csv(source_name)
df = raw.copy(deep=True)

print("Source:", source_name)
print("Raw shape:", raw.shape)
print("Columns:", raw.columns.tolist())

display(raw.head())
display(raw.dtypes.to_frame("raw_dtype"))
display(raw.describe(include="all").T)

# Preserve raw values. Parse into separate working columns so conversion
# failures can be counted without overwriting the source columns.
df["date_parsed"] = pd.to_datetime(raw["date"], errors="coerce")
df["price_numeric"] = pd.to_numeric(raw["price"], errors="coerce")

original_missing = int(raw.isna().sum().sum())
price_conversion_failures = int(
    (raw["price"].notna() & df["price_numeric"].isna()).sum()
)
date_conversion_failures = int(
    (raw["date"].notna() & df["date_parsed"].isna()).sum()
)

checks = {
    "rows": len(raw),
    "columns": len(raw.columns),
    "original_missing_cells": original_missing,
    "exact_duplicate_rows": int(raw.duplicated().sum()),
    "duplicate_date_item_city_records": int(
        raw.duplicated(subset=["date", "item", "city"], keep=False).sum()
    ),
    "non_numeric_price_conversion_failures": price_conversion_failures,
    "non_positive_numeric_prices": int((df["price_numeric"].dropna() <= 0).sum()),
    "unparseable_date_conversion_failures": date_conversion_failures,
    "distinct_dates_in_raw_data": int(raw["date"].nunique(dropna=True)),
    "distinct_cities": int(raw["city"].nunique(dropna=True)),
}
display(pd.Series(checks, name="result").to_frame())

print("Missing values in the original input, by column:")
display(raw.isna().sum().to_frame("original_missing"))

print("Rows with originally missing values:")
display(raw.loc[raw.isna().any(axis=1)])

print("Rows with non-numeric, non-missing price values:")
display(raw.loc[raw["price"].notna() & df["price_numeric"].isna()])

print("Exact duplicate rows (all occurrences):")
display(raw.loc[raw.duplicated(keep=False)].sort_values(
    by=["date", "item", "city"], na_position="last"
))

print("Rows participating in repeated date-item-city keys:")
key_columns = ["date", "item", "city"]
display(raw.loc[raw.duplicated(subset=key_columns, keep=False)].sort_values(
    by=key_columns, na_position="last"
))

print("Rows with non-positive numeric prices:")
display(df.loc[df["price_numeric"] <= 0, raw.columns.tolist()])

print("Item labels and counts:")
display(raw["item"].value_counts(dropna=False).to_frame("count"))

# This comparison only identifies labels that become equal after trimming
# spaces and ignoring case. It does not recode the original item column.
item_comparison = (
    raw["item"].dropna().astype(str).str.strip().str.casefold()
)
print("Item labels after trim/case-fold comparison (diagnostic only):")
display(item_comparison.value_counts().to_frame("count"))

print("Unit labels:")
display(raw["unit"].value_counts(dropna=False).to_frame("count"))

print("Source/data labels:")
display(raw["data_label"].value_counts(dropna=False).to_frame("count"))

# The expected count is a diagnostic benchmark, not proof that a record
# should be created or deleted.
expected_per_item = raw["city"].nunique() * raw["date"].nunique()
print("Expected rows per item if every date-city combination is present:",
      expected_per_item)
display(raw["item"].value_counts(dropna=False).to_frame("actual_rows"))

print("Numeric price summary (conversion failures excluded, not imputed):")
display(df["price_numeric"].describe().to_frame("price_numeric"))

# Before/after comparison: the raw table is unchanged. The working copy
# adds parsed diagnostic columns; it does not drop or impute observations.
summary = pd.DataFrame({
    "raw": [
        len(raw),
        int(raw.isna().sum().sum()),
        int(raw.duplicated().sum()),
    ],
    "working_copy": [
        len(df),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
    ],
}, index=["rows", "missing_cells_in_all_columns", "exact_duplicate_rows"])
summary["change"] = summary["working_copy"] - summary["raw"]
display(summary)

print("No rows were deleted and no values were imputed.")
print("Inspect the flagged records before making any correction.")


Source: teaching_prices_messy.csv
Raw shape: (97, 6)
Columns: ['date', 'item', 'city', 'price', 'unit', 'data_label']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


,raw_dtype
date,object
item,object
city,object
price,object
unit,object
data_label,object


,count,unique,top,freq
date,97,12,2026-01-08,9
item,97,5,Staple A,24
city,97,2,City 1,49
price,96,70,202.5,5
unit,97,1,PKR per kg (illustrative),97
data_label,97,1,ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,97


,result
rows,97
columns,6
original_missing_cells,1
exact_duplicate_rows,1
duplicate_date_item_city_records,2
non_numeric_price_conversion_failures,1
non_positive_numeric_prices,0
unparseable_date_conversion_failures,0
distinct_dates_in_raw_data,12
distinct_cities,2


Missing values in the original input, by column:


... (21 outputs left out)

Numeric price summary (conversion failures excluded, not imputed):


,price_numeric
count,95.000000
mean,305.621053
std,1008.261424
min,99.500000
25%,135.500000
50%,200.000000
75%,301.000000
max,9999.000000


,raw,working_copy,change
rows,97,97,0
missing_cells_in_all_columns,1,3,2
exact_duplicate_rows,1,1,0


No rows were deleted and no values were imputed.
Inspect the flagged records before making any correction.
